# M06-02 — Cache y particionado

[← Anterior](02-lab-explain-dag.ipynb) · [Siguiente →](../M07-persistencia-datos/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Dos ideas **distintas** (no las mezcles):

1. **Cache** — “guarda este resultado en memoria para no repetir el plan”. Se llena con una **acción**, no al escribir `.cache()`. La prueba **no** es que el `count` salga 9016 dos veces (eso es normal). La prueba es Storage **0 → 0 → 1 → 1 → 0**.
2. **`repartition`** — baraja filas en **memoria** a N trozos. No crea carpetas en disco (eso es `write.partitionBy` en M07) ni es el `partitionBy` de las ventanas (M05).

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M06-02-cache-particionado.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


## Léelo antes (si no, “no demuestra nada”)

### Palabras que se parecen y no son lo mismo

| Lo que escribes | Dónde vive | Para qué |
|-----------------|------------|----------|
| `Window.partitionBy("customer_id")` (M05) | receta de la ventana | recortar el **vecindario** (por cliente) |
| `df.repartition(12, col("order_month"))` | **memoria**, ahora | barajar a 12 trozos (shuffle) |
| `write.partitionBy("order_month")` (M07) | **disco**, carpetas | `order_month=2024-01/` |
| `df.cache()` | memoria, **después** de una acción | no repetir un plan caro |

### Cómo se demuestra el cache (no mires el 9016)

El número de filas **tiene** que ser igual con cache y sin él. Cache no cambia el resultado; cambia **de dónde** sale la segunda lectura.

| Momento | Storage (pestaña o `n_en_storage()`) | Qué significa |
|---------|--------------------------------------|---------------|
| DataFrame creado | **0** | no hay nada guardado |
| Acabas de escribir `.cache()` | **0** | solo **marcó**; aún no calculó |
| 1.er `count` | **1** | esa acción **llenó** memoria |
| 2.º `count` | **1** | reutiliza; no soltó |
| `unpersist()` | **0** | suelta (si no, el Codespace se llena) |

Los **segundos** del cronómetro en local a veces no se inmutan. Si el reloj miente, Storage no.

Spark UI: Ports **4040** → pestaña **Storage**. Jobs: el 1.er count hace Scan+Filter; el 2.º debería verse más corto / *InMemory*.

Necesitas `fact_lines` (M03). No subas de 8 copias: OOM.


### Paso 1 — El fact de siempre + qué es una partición

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Una **partición** (aquí) es un **trozo de trabajo en memoria**, no una carpeta. `getNumPartitions()` dice en cuántos trozos está **ahora** este DataFrame.

Leemos las 1980 líneas. Aún no hay cache.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `filas fact_lines 1980`. Particiones **> 1**. Storage **0**.

**Por qué este paso.** Si sale PATH error, no es este lab: falta M03-02 / pipeline.

**Si no sale.** 1980 no sale → regenera staging. Storage ≠ 0 → `spark.catalog.clearCache()` y reejecuta.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, lit
from time import perf_counter

spark = get_spark("novashop-m06")
spark.catalog.clearCache()  # por si una celda anterior dejó basura

def n_en_storage():
    # Cuántos RDD hay en memoria (= pestaña Storage). 0 vacío, ≥1 hay cache lleno.
    return int(spark.sparkContext._jsc.getPersistentRDDs().size())

base = spark.read.parquet(str(STAGING / "fact_lines"))
print("filas fact_lines", base.count())                 # 1980
print("particiones ahora", base.rdd.getNumPartitions())  # > 1 en local[*]
print("Storage (tiene que ser 0)", n_en_storage())


### Paso 2 — Por qué apilamos 8 copias (lupa, no producción)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

1980 filas (como las 20 de la teoría) son tan pocas que el 1.er y el 2.º `count` duran igual y parece que el cache “no hace nada”.

`unionByName` **apila** la misma tabla 8 veces: 1980 × 8 = **15840** filas. `_copy` marca de qué copia sale cada fila. No es un patrón de pipeline; es para que Storage tenga algo que mostrar y el job dure un poco.

`range(7)` + la base con `_copy = -1` → 8 copias en total.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** **15840**. Storage **0**. Varias particiones.

**Por qué este paso.** Si cuentas 1980, el bucle no se ejecutó (reusas `base` en vez de `xl`).


In [ ]:
xl = base.withColumn("_copy", lit(-1))
for i in range(7):
    xl = xl.unionByName(base.withColumn("_copy", lit(i)))

print("filas apiladas", xl.count())                       # 15840
print("particiones tras el union", xl.rdd.getNumPartitions())
print("Storage sigue", n_en_storage())                    # 0: aún no hay cache


### Paso 3 — Dos counts **fríos** (sin cache): el plan se repite

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Sin cache, cada `count` vuelve a leer Parquet + unions + filtro cobrable.

`timed_count` mide reloj **y** imprime filas. Los dos tiempos son del **mismo orden**. El número **9016** las dos veces no demuestra cache: demuestra que el filtro es el mismo (1127 cobrables × 8).

Storage sigue en 0: no hemos marcado nada.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Las dos veces **n = 9016**. Storage **0** y **0**. Tiempos parecidos (si no, mira Jobs: son dos jobs del mismo estilo, Scan otra vez).

**Por qué este paso.** Frío = cada acción rehace el plan. Aún no hay atajo.


In [ ]:
def timed_count(df, label):
    t0 = perf_counter()
    n = df.count()
    print(label, "n =", n, "s =", round(perf_counter() - t0, 3),
          "Storage =", n_en_storage())
    return n

# Filtro cobrable AQUÍ (no dentro de timed_count: si no, no se entiende qué cacheas)
billable = xl.where(col("is_billable"))

timed_count(billable, "1er count frío")
timed_count(billable, "2º count frío")


### Paso 4 — `.cache()` **solo marca** — Storage sigue vacío

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

`cache()` quiere decir: “la **próxima** acción, guarda el resultado”. Hasta que no haya `count`/`show`, Storage = 0.

Ejecuta **esta** celda y **párate**. Mira UI → Storage (vacío) y el print. No lances el count todavía.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Storage **0**. `storageLevel` ya dice MEMORY (eso es la marca, no el llenado).

**Por qué este paso.** Si aquí ya ves 1, el kernel tenía un cache viejo: `clearCache()` y desde el paso 1.


In [ ]:
warm = billable.cache()
print("acabo de escribir cache(); Storage =", n_en_storage())  # 0
print("storageLevel (la intención, no los datos):", warm.storageLevel)
print("PARA AQUÍ. Storage tiene que seguir vacío.")


### Paso 5 — El 1.er count llena; el 2.º reutiliza

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Ahora sí: la primera acción **materializa**. La segunda debería leer memoria.

El 9016 **otra vez** es correcto: cache no cambia el resultado. Lo que cambia es Storage 0→**1** y, en `explain`, *InMemoryTableScan* / *InMemoryRelation*.

No hagas `unpersist` en esta celda: si no, al terminar Storage vuelve a 0 y “no se ve nada”.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** 1º: n=9016, Storage **1**. 2º: n=9016, Storage **1**. En el explain aparece *InMemory*. El reloj del 2º no empeora (a veces es igual de corto: fíate de Storage).

**Por qué este paso.** Dos 9016 no son la demo. Un Storage que pasa a 1 sí.


In [ ]:
timed_count(warm, "1º count (LLENA el cache)")
print("   → Storage tiene que ser 1. Mira también UI → Storage.")

timed_count(warm, "2º count (lee cache, mismo n)")
print("   → Storage sigue 1 (no lo soltó)")

print("--- Plan del 2º count: busca InMemoryTableScan / InMemoryRelation ---")
warm.explain("formatted")


### Paso 6 — `unpersist` suelta (otra celda, siempre)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Si no sueltas, el Codespace se queda con el fact×8 en memoria. `unpersist()` vacía Storage.

Jupyter a veces pinta `DataFrame[...]` porque `unpersist` **devuelve** el DataFrame: ignóralo, o deja el `print` al final.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Storage **0**. UI → Storage vacío otra vez.

**Por qué este paso.** Si vuelves a `warm.count()` ahora, **recalcula** (frío) y, como ya no está cacheado, Storage sigue 0.


In [ ]:
warm.unpersist()
print("después de unpersist, Storage =", n_en_storage())  # 0
print("listo")


## Prueba tú — Cache sin acción = Storage vacío

No copies y listo: **cambia** lo que indica el texto y mira si cuadra con **Qué tienes que ver**.

Sin pegar el paso 5: `otra = billable.cache()` y `print(n_en_storage())`. Luego un `count` y otra vez el print. `unpersist` al terminar.

**Qué tienes que ver.** **0**, luego **≥ 1**, luego **0**. Escríbelo en Markdown. Eso es todo el lab de cache.


In [ ]:
otra = billable.cache()
print("solo cache(), Storage =", n_en_storage())  # 0
_ = otra.count()
print("tras count, Storage =", n_en_storage())    # ≥ 1
otra.unpersist()
print("tras unpersist, Storage =", n_en_storage())  # 0


## Parte B — `repartition` (esto **no** es cache)

Ya sabes llenar y soltar memoria. Ahora: **cómo está troceado** el DataFrame en RAM.

`repartition(12, col("order_month"))` **baraja** (shuffle) hacia 12 particiones, una intención por mes. Sirve de cara a **escribir** en M07. No crea las carpetas todavía.

Si omites el `12` y pones solo la columna, Spark 3.5 usa **200** particiones por defecto: inútil aquí.


### Paso 7 — 12 particiones por mes (shuffle en memoria)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Vuelve a cachear `billable` un momento (el paso 6 lo soltó) o reparte sobre `billable` directo.

`getNumPartitions() == 12`. El `groupBy("order_month")` debe listar **12** meses de 2024.

Esto **no** es `Window.partitionBy`. Esto **no** es `write.partitionBy`.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `particiones 12`. Doce filas de meses `2024-01` … `2024-12`.

**Por qué este paso.** Si ves 200, llamaste `repartition(col("order_month"))` **sin** el 12.


In [ ]:
# El cache del paso 5 ya se soltó; no hace falta para contar particiones
by_month = billable.repartition(12, col("order_month"))
print("particiones", by_month.rdd.getNumPartitions())  # 12

(
    by_month.groupBy("order_month")
    .count()
    .orderBy("order_month")
    .show()
)


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Markdown con la tabla Storage de tu ejecución: 0 (tras cache sin count), 1 (tras 1.er count), 1 (tras 2.º), 0 (unpersist).

Una frase: el 9016 igual las dos veces **no** es la prueba.

`getNumPartitions()` tras el paso 7 = **12**.


## Mejora — coalesce(1) vs repartition(1)

Los dos dejan 1 partición, pero no igual de caro:

- `repartition(1)` **siempre** shufflea (`Exchange` en el plan).
- `coalesce(1)` **reduce** trozos sin un shuffle amplio.

Útil para un único fichero de entrega; mal hábito si lo pones en mitad del pipeline (M07). Ejecuta los dos `explain` y copia la línea donde uno dice *Exchange* y el otro no.

Si te atasca, el código está en la celda siguiente.


In [ ]:
print("===== coalesce(1) (sin shuffle amplio) =====")
billable.coalesce(1).explain("formatted")
print("===== repartition(1) (shuffle: busca Exchange) =====")
billable.repartition(1).explain("formatted")
print("particiones coalesce", billable.coalesce(1).rdd.getNumPartitions())
print("particiones repartition", billable.repartition(1).rdd.getNumPartitions())


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| Los dos count salen 9016 y “no demuestra nada” | El número tiene que ser igual | Mira Storage 0→1→1→0, no el 9016 |
| Storage vacío tras cache() | No hubo acción, o unpersist en la misma celda | count en celda aparte; unpersist después |
| Storage = 1 al escribir cache() | Cache viejo en el kernel | `spark.catalog.clearCache()` |
| OOM / el kernel muere | Demasiadas copias + cache | Quédate en 8; `unpersist` |
| 1980 o 200 particiones en el paso 7 | `repartition` sin `12` | `repartition(12, col("order_month"))` |
| UI en 4041 / vacía | Otra SparkSession ocupó 4040 | `spark.stop()`; Ports 4040 |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M07 — teoría](../M07-persistencia-datos/01-teoria.ipynb).
